# Bank Transaction Fraud Detection (By Joy Njuguna (677112) and Tamia Wambugu(674367) and Igiraneza Yvannelle(674623))
This notebook demonstrates building and evaluating three different models for fraud detection using a dataset of over 5,000 transactions:
1. **Random Forest** (Supervised Machine Learning)
2. **K-Means Clustering** (Unsupervised Anomaly Detection)
3. **TensorFlow Deep Neural Network** (Supervised Deep Learning)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# 1. Generate a synthetic dataset of 6,000 transactions
n_samples = 6000
data = {
    'transaction_id': [f'TX_{i:06d}' for i in range(n_samples)],
    'amount': np.random.exponential(scale=120, size=n_samples),
    'old_balance': np.random.uniform(10, 100000, size=n_samples),
    'hour_of_day': np.random.randint(0, 24, size=n_samples),
    'age_of_account_days': np.random.randint(1, 1800, size=n_samples),
    'is_merchant': np.random.choice([0, 1], size=n_samples, p=[0.75, 0.25]),
    'device_type': np.random.choice(['mobile', 'desktop', 'tablet'], size=n_samples, p=[0.6, 0.3, 0.1])
}

df = pd.DataFrame(data)
df['new_balance'] = df['old_balance'] - df['amount']

# Heuristic to simulate real-world fraud indicators
fraud_prob = (
    (df['amount'] > 1500) * 0.4 +
    ((df['hour_of_day'] >= 1) & (df['hour_of_day'] <= 4) & (df['amount'] > 300)) * 0.5 +
    (df['age_of_account_days'] < 15) * (df['amount'] > 500) * 0.6 +
    0.005
)
df['is_fraud'] = np.random.binomial(1, np.minimum(fraud_prob, 0.95))

print(f"Dataset generated with {df.shape[0]} records.")
print(f"Fraud cases: {df['is_fraud'].sum()} ({df['is_fraud'].mean()*100:.2f}% of total)")
display(df.head())


Dataset generated with 6000 records.
Fraud cases: 68 (1.13% of total)


,transaction_id,amount,old_balance,hour_of_day,age_of_account_days,is_merchant,device_type,new_balance,is_fraud
0,TX_000000,56.312171,64829.212838,21,1717,0,mobile,64772.900667,0
1,TX_000001,361.214572,17246.912347,19,1364,1,desktop,16885.697775,0
2,TX_000002,158.009483,87240.732390,6,441,0,desktop,87082.722907,0
3,TX_000003,109.553106,61315.492745,2,617,0,mobile,61205.939638,0
4,TX_000004,20.354984,15728.816317,4,226,0,desktop,15708.461332,0


## Preprocessing & Cleaning

In [2]:
# Check for missing values
print("Missing Values:\n", df.isnull().sum())

# One-Hot Encoding for categorical device variable
df_encoded = pd.get_dummies(df, columns=['device_type'], drop_first=True)

# Select feature columns
feature_cols = ['amount', 'old_balance', 'new_balance', 'hour_of_day', 'age_of_account_days', 'is_merchant'] + [col for col in df_encoded.columns if 'device_type_' in col]
X = df_encoded[feature_cols]
y = df_encoded['is_fraud']

# Split training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


Missing Values:
 transaction_id         0
amount                 0
old_balance            0
hour_of_day            0
age_of_account_days    0
is_merchant            0
device_type            0
new_balance            0
is_fraud               0
dtype: int64


## Model Evaluation Function

In [3]:
def evaluate_model(name, y_true, y_pred, y_prob=None):
    print(f"=== {name} Performance ===")
    print(classification_report(y_true, y_pred))
    if y_prob is not None:
        print(f"ROC-AUC Score: {roc_auc_score(y_true, y_prob):.4f}\n")


## 1. Random Forest Classifier

In [4]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train_scaled, y_train)

rf_preds = rf.predict(X_test_scaled)
rf_probs = rf.predict_proba(X_test_scaled)[:, 1]

evaluate_model("Random Forest", y_test, rf_preds, rf_probs)


=== Random Forest Performance ===
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1186
           1       0.50      0.07      0.12        14

    accuracy                           0.99      1200
   macro avg       0.74      0.54      0.56      1200
weighted avg       0.98      0.99      0.98      1200

ROC-AUC Score: 0.7330



## 2. K-Means Clustering (Anomaly Detection)

In [5]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans.fit(X_train_scaled)

# Measure distances to the closest centroid to flag outliers
centroids = kmeans.cluster_centers_
labels = kmeans.predict(X_test_scaled)
distances = np.sqrt(np.sum((X_test_scaled - centroids[labels])**2, axis=1))

# Threshold defined at 95th percentile of distances
threshold = np.percentile(distances, 95)
kmeans_preds = (distances > threshold).astype(int)

evaluate_model("K-Means Anomaly Detection", y_test, kmeans_preds)


=== K-Means Anomaly Detection Performance ===
              precision    recall  f1-score   support

           0       0.99      0.95      0.97      1186
           1       0.08      0.36      0.14        14

    accuracy                           0.95      1200
   macro avg       0.54      0.66      0.55      1200
weighted avg       0.98      0.95      0.96      1200



## 3. TensorFlow Deep Neural Network

In [6]:
# Calculate class weights to deal with class imbalance
neg, pos = np.bincount(y_train)
total = neg + pos
weight_for_0 = (1 / neg) * (total / 2.0)
weight_for_1 = (1 / pos) * (total / 2.0)
class_weight = {0: weight_for_0, 1: weight_for_1}

# Build Sequential Model
model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

# Train Model
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.1,
    epochs=15,
    batch_size=64,
    class_weight=class_weight,
    verbose=1
)

# Predict
tf_probs = model.predict(X_test_scaled).flatten()
tf_preds = (tf_probs > 0.5).astype(int)

evaluate_model("TensorFlow Neural Network", y_test, tf_preds, tf_probs)


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.9303 - auc: 0.4052 - loss: 0.7872 - val_accuracy: 0.9417 - val_auc: 0.5303 - val_loss: 0.5059
Epoch 2/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9083 - auc: 0.5340 - loss: 0.7021 - val_accuracy: 0.9000 - val_auc: 0.6890 - val_loss: 0.5343
Epoch 3/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8796 - auc: 0.6810 - loss: 0.6453 - val_accuracy: 0.8625 - val_auc: 0.7585 - val_loss: 0.5442
Epoch 4/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8389 - auc: 0.7444 - loss: 0.6130 - val_accuracy: 0.8354 - val_auc: 0.7794 - val_loss: 0.5296
Epoch 5/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8234 - auc: 0.7673 - loss: 0.5837 - val_accuracy: 0.8021 - val_auc: 0.7950 - val_loss: 0.5242
Epoch 6/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7840 - auc: 0.7707 - loss: 0.5689 - val_accuracy: 0.7854 - val_auc: 0.7929 - val_loss: 0.5247
Epoch 7/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step